In [7]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
import numpy as np

# Function to evaluate a model using cross-validation
def evaluate_model(model, X, y, num_folds=5):
    mse_scores = -cross_val_score(model, X, y, cv=num_folds, scoring='neg_mean_squared_error')
    r2_scores = cross_val_score(model, X, y, cv=num_folds, scoring='r2')
    return mse_scores.mean(), r2_scores.mean()

# Function to evaluate models with different alpha values
def evaluate_models(X, y, alpha_values):
    cross_val_results = {}
    for alpha in alpha_values:
        # Ridge
        ridge = Ridge(alpha=alpha)
        mse_ridge, r2_ridge = evaluate_model(ridge, X, y)
        cross_val_results[f'Ridge_alpha_{alpha}'] = {'MSE': mse_ridge, 'R^2': r2_ridge}

        # Lasso
        lasso = Lasso(alpha=alpha)
        mse_lasso, r2_lasso = evaluate_model(lasso, X, y)
        cross_val_results[f'Lasso_alpha_{alpha}'] = {'MSE': mse_lasso, 'R^2': r2_lasso}

        # Elastic Net
        elastic_net = ElasticNet(alpha=alpha)
        mse_elastic_net, r2_elastic_net = evaluate_model(elastic_net, X, y)
        cross_val_results[f'ElasticNet_alpha_{alpha}'] = {'MSE': mse_elastic_net, 'R^2': r2_elastic_net}

    # Also evaluate Linear Regression
    mse_lr, r2_lr = evaluate_model(LinearRegression(), X, y)
    cross_val_results['Linear Regression'] = {'MSE': mse_lr, 'R^2': r2_lr}

    return cross_val_results

# Load the datasets
data_hy = pd.read_csv('water_HY.csv')
data_wqi = pd.read_csv('AKH_WQI.csv')

# Convert 'Date' and 'Time' to datetime and extract features for HY dataset
data_hy['DateTime'] = pd.to_datetime(data_hy['Date'] + ' ' + data_hy['Time'])
data_hy['Year'] = data_hy['DateTime'].dt.year
data_hy['Month'] = data_hy['DateTime'].dt.month
data_hy['Day'] = data_hy['DateTime'].dt.day
data_hy['Hour'] = data_hy['DateTime'].dt.hour
data_hy['Weekday'] = data_hy['DateTime'].dt.weekday
data_hy.drop(['Date', 'Time', 'DateTime'], axis=1, inplace=True)

# Assuming 'Muc_nuoc' is the target for HY dataset
X_hy = data_hy.drop(['Muc_nuoc'], axis=1)
y_hy = data_hy['Muc_nuoc']

# Assuming 'WQI' is the target for WQI dataset
X_wqi = data_wqi.drop(['WQI'], axis=1)
y_wqi = data_wqi['WQI']

# Alpha values for Ridge, Lasso, and Elastic Net
alpha_values = [0.001, 0.01, 0.1, 1]

# Evaluate models for HY dataset
results_hy = evaluate_models(X_hy, y_hy, alpha_values)

# Evaluate models for WQI dataset
results_wqi = evaluate_models(X_wqi, y_wqi, alpha_values)

# Convert the results to DataFrame for better visualization
results_df_hy = pd.DataFrame(results_hy).transpose()
results_df_hy['Dataset'] = 'HY'
results_df_wqi = pd.DataFrame(results_wqi).transpose()
results_df_wqi['Dataset'] = 'WQI'

# Combine results for both datasets
combined_results_df = pd.concat([results_df_hy.reset_index(), results_df_wqi.reset_index()])
combined_results_df.rename(columns={'index': 'Model'}, inplace=True)

print(combined_results_df)


                     Model         MSE       R^2 Dataset
0        Ridge_alpha_0.001    2.757774  0.994289      HY
1        Lasso_alpha_0.001    2.753395  0.994297      HY
2   ElasticNet_alpha_0.001    2.755362  0.994293      HY
3         Ridge_alpha_0.01    2.757733  0.994289      HY
4         Lasso_alpha_0.01    2.715164  0.994366      HY
5    ElasticNet_alpha_0.01    2.734528  0.994332      HY
6          Ridge_alpha_0.1    2.757318  0.994290      HY
7          Lasso_alpha_0.1    2.380066  0.994913      HY
8     ElasticNet_alpha_0.1    2.545548  0.994662      HY
9            Ridge_alpha_1    2.753216  0.994299      HY
10           Lasso_alpha_1    2.055015  0.993873      HY
11      ElasticNet_alpha_1    2.047396  0.994880      HY
12       Linear Regression    2.757779  0.994289      HY
0        Ridge_alpha_0.001  521.345242  0.209973     WQI
1        Lasso_alpha_0.001  521.378332  0.209929     WQI
2   ElasticNet_alpha_0.001  521.370668  0.209946     WQI
3         Ridge_alpha_0.01  521

Nhận Xét

In [ ]:
# Tất cả các mô hình đều có điểm R^2 rất cao, cho thấy chúng khớp tốt với dữ liệu.
# Lasso và Elastic Net Regression với alpha = 1 cho kết quả tốt nhất về MSE so với các mô hình khác.
# Việc tinh chỉnh siêu tham số alpha có tác động đáng kể đến hiệu suất của mô hình, đặc biệt là đối với Lasso và Elastic Net Regression.